# Self-supervised pretraining with AstroLens AstroPT

A tutorial-scale example for `astrolens.models.astropt.AstroPT` (Smith et al.,
2024, https://arxiv.org/abs/2405.14930), using
[`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10)
(Galaxy10 DECals, 17,736 galaxies, 10 discrete morphology classes).

Unlike `Linformer` and `GCNN`, AstroPT is a self-supervised backbone: it is
pretrained on unlabeled images with a causal next-patch prediction objective,
then adapted to a downstream task. This notebook does both steps at tutorial
scale — a few pretraining epochs, then a LoRA finetune (Hu et al., 2021,
https://arxiv.org/abs/2106.09685) of a classification head on top, following
the reference model's own `scripts/finetune.py` recipe: freeze the pretrained
weights, train only injected low-rank adapters and the head.

The original work pretrains at far larger scale (hundreds of millions of
DESI Legacy Survey images, models up to ~2.1B parameters) and publishes
checkpoints on Hugging Face (`Smith42/astroPT`, `Smith42/astroPT_v2.0`). For
real downstream performance, start from one of those checkpoints rather than
pretraining from scratch here; this notebook demonstrates the mechanism, not
the paper's scale.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [ ]:
!pip install -q datasets torchvision scikit-learn

## Imports

In [ ]:
import io

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import train_test_split

import astrolens

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and split 70/10/20

Labels are only needed for the finetuning stage; pretraining uses the
images alone.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
CLASS_NAMES = [
    "disturbed",
    "merging",
    "round_smooth",
    "in_between_round_smooth",
    "cigar_shaped_smooth",
    "barred_spiral",
    "unbarred_tight_spiral",
    "unbarred_loose_spiral",
    "edge_on_no_bulge",
    "edge_on_with_bulge",
]
NUM_CLASSES = len(CLASS_NAMES)

gz10 = load_dataset("UniverseTBD/mmu_gz10", split="train")
labels = gz10["gz10_label"]

train_idx, rest_idx = train_test_split(
    range(len(gz10)), train_size=0.7, stratify=labels, random_state=0
)
val_idx, test_idx = train_test_split(
    rest_idx,
    train_size=1 / 3,  # 1/3 of the remaining 30% -> 10% val, 20% test
    stratify=[labels[i] for i in rest_idx],
    random_state=0,
)

# per-channel mean/std computed from a 2000-image sample of the gz10 train split
IMAGE_MEAN = [0.1675, 0.1625, 0.1586]
IMAGE_STD = [0.1288, 0.1178, 0.1109]

train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)


class GZ10Dataset(Dataset):
    """Map-style wrapper around an index subset of the HF split, applying transform lazily."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split = hf_split
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        example = self.hf_split[self.indices[i]]
        image = Image.open(io.BytesIO(example["rgb_image"]["bytes"])).convert("RGB")
        return self.transform(image), example["gz10_label"]


train_dataset = GZ10Dataset(gz10, train_idx, train_transform)
val_dataset = GZ10Dataset(gz10, val_idx, eval_transform)
test_dataset = GZ10Dataset(gz10, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset), len(test_dataset)

## Pretrain the backbone

A small config (`patch_size=16` -> 196 patches at 224x224, `dim=192`,
`depth=6`, `heads=3`) so a few epochs run in reasonable time on one GPU.
`num_classes` is left unset: `forward()` then returns next-patch predictions,
and `loss()` computes the autoregressive (Huber) pretraining objective.

In [ ]:
PATCH_SIZE = 16
DIM = 192
DEPTH = 6
HEADS = 3

backbone = astrolens.create_model(
    "astropt",
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
).to(device)

sum(p.numel() for p in backbone.parameters())

In [ ]:
PRETRAIN_EPOCHS = 5
PRETRAIN_LR = 3e-4

optimizer = torch.optim.AdamW(backbone.parameters(), lr=PRETRAIN_LR)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    backbone.train()
    total_loss, count = 0.0, 0
    for images, _ in train_loader:
        images = images.to(device)
        loss = backbone.loss(images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        count += images.size(0)

    print(f"epoch {epoch:02d} pretrain_loss={total_loss / count:.4f}")

## Finetune with LoRA

A second model instance adds `num_classes` (a classification head) and
`lora_r` (low-rank adapters injected into each attention block's qkv
projection). Its weights are copied from the pretrained backbone with
`strict=False`, since it has extra `lora_*`/`head.*` parameters the
pretrained model doesn't. `mark_only_lora_as_trainable()` then freezes
everything else, matching the reference model's `finetune.py`: only the
LoRA adapters and task head are updated, the pretrained backbone stays
fixed.

In [ ]:
LORA_R = 8

model = astrolens.create_model(
    "astropt",
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
    num_classes=NUM_CLASSES,
    lora_r=LORA_R,
).to(device)
model.load_state_dict(backbone.state_dict(), strict=False)
model.mark_only_lora_as_trainable()

sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
FINETUNE_EPOCHS = 20
FINETUNE_LR = 1e-3

criterion = nn.CrossEntropyLoss()
finetune_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=FINETUNE_LR
)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                finetune_optimizer.zero_grad()
                loss.backward()
                finetune_optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += images.size(0)

    return total_loss / count, correct / count


for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(
        f"epoch {epoch:02d} train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

## Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

## Next steps

- Raise `PRETRAIN_EPOCHS` or train on more data for a more useful backbone;
  five epochs on 17k images is enough to demonstrate the mechanism, not to
  reach the reference model's reported performance.
- Try a plain linear probe instead of LoRA (`lora_r=0`, freeze everything
  except `model.head`) for a cheaper, lower-capacity alternative.
- Try `AstroPT(..., spiral=True)` for spiral instead of raster patch ordering,
  or `forward_features(images, draw_from_centre=True)` for mid-layer
  embeddings — both are choices from the reference model's own scaling study.
- For paper-scale pretraining or published checkpoints, see
  [`Smith42/astropt`](https://github.com/Smith42/astropt) and
  [`Smith42/astroPT`](https://huggingface.co/Smith42/astroPT) on Hugging Face.